# Day 05 · Deduplication and deterministic records

Day 04 left **775 clean lines**, but some lines appear twice. Today we keep exactly **one row per `line_id`**. The rule for "which one" must give the **same answer every time** it runs (deterministic).

| Task | Concept |
|---|---|
| 1 | Grain and business key: what does one row mean? |
| 2 | Find the duplicates |
| 3 | `distinct` vs `dropDuplicates`, and what goes wrong with the wrong key |
| 4 | Keep one row per key with a window and `row_number` |
| 5 | Corrections: keep the **latest** version, with a tie-break |
| 6 | Publish the clean lines and check every raw line is accounted for |

**Needs Day 04 first:** this notebook reads Day 04's output, `scratch/day04/clean_lines`. Run on the **pyspark-course** cluster.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import json

RAW     = "/Volumes/dbw_pyspark_course/retail/raw"
SCRATCH = "/Volumes/dbw_pyspark_course/retail/scratch"
OUT     = f"{SCRATCH}/day05"     # today's outputs

manifest = json.load(open(f"{RAW}/manifest.json"))

# Day 04 output: the 775 lines that passed every rule
clean_lines = spark.read.parquet(f"{SCRATCH}/day04/clean_lines")
print("lines from Day 04:", clean_lines.count())

### Task 1 · Grain and business key

The **grain** says what one row means. The **business key** is the column (or columns) that identifies that row.

| Table | One row = | Business key |
|---|---|---|
| customers | one customer | `customer_id` |
| products | one product | `product_id` |
| stores | one store | `store_id` |
| orders | one order | `order_id` |
| order_items | **one order line** | **`line_id`** |
| returns | one return event | `return_id` |

**Problem:** Show that `order_id` is **not** a key for `order_items`, because one order can have several lines.

**Expected:**
- Order `O0001` has **3** different lines.
- **230** of the 300 orders have more than one line.

So the same `order_id` repeating is normal and **not** a duplicate. Only a repeated `line_id` is a duplicate.

In [ ]:
# Count the different lines in each order
lines_per_order = clean_lines.groupBy("order_id").agg(F.countDistinct("line_id").alias("lines_in_order"))

lines_per_order.orderBy("order_id").show(5)
print("orders with more than one line:", lines_per_order.filter(F.col("lines_in_order") > 1).count())

### Task 2 · Find the duplicates

**Problem:**
1. How many rows are **exact copies** of another row, identical in every column?
2. Which `line_id` values appear more than once?

**Expected:**
1. 775 rows, but only 767 distinct rows, so there are **8 exact duplicates**.
2. **8** line_ids appear twice: `L0001_1`, `L0001_2`, `L0001_3`, `L0002_1`, `L0002_2`, `L0003_1`, `L0003_2` and `L0004_1`.

In [ ]:
total_rows    = clean_lines.count()
distinct_rows = clean_lines.distinct().count()   # distinct() compares ALL columns

print("rows            :", total_rows)
print("distinct rows   :", distinct_rows)
print("exact duplicates:", total_rows - distinct_rows)

In [ ]:
# line_id values that appear more than once
duplicate_ids = clean_lines.groupBy("line_id").count().filter(F.col("count") > 1)
duplicate_ids.orderBy("line_id").show()

### Task 3 · `distinct` vs `dropDuplicates`

- `distinct()` removes rows that are identical in **every** column.
- `dropDuplicates(["col"])` keeps **one row per value** of the column(s) you list, and ignores the other columns.

**Problem:** Compare `distinct()`, `dropDuplicates(["line_id"])` and `dropDuplicates(["order_id"])`.

**Expected:**
- `distinct()` → **767**
- `dropDuplicates(["line_id"])` → **767**. Same here, because our duplicates are exact copies.
- `dropDuplicates(["order_id"])` → **300**. **467 real lines are lost!** It keeps one random line per order. Wrong key = lost data.

In [ ]:
print("distinct()                  :", clean_lines.distinct().count())
print("dropDuplicates(['line_id']) :", clean_lines.dropDuplicates(["line_id"]).count())
print("dropDuplicates(['order_id']):", clean_lines.dropDuplicates(["order_id"]).count())

### Task 4 · Keep one row per key with a window and `row_number`

A **window** looks at groups of related rows without merging them:
- `partitionBy("line_id")` puts all rows with the same `line_id` in one group.
- `orderBy(...)` sorts the rows inside each group.
- `row_number()` numbers them 1, 2, 3… in that order.

Keep row **1** and set the rest aside.

**Problem:** Number the rows within each `line_id`, keep `rn = 1`, and keep the removed copies for audit.

**Expected:**
- `L0001_1` shows two rows, numbered **1** and **2**.
- **767 kept** and **8 removed**.

Here we sort by `order_id, product_id`, as the course's starter does. Our duplicates are identical, so it doesn't matter which copy wins. Task 5 shows when it **does** matter.

In [ ]:
# Group by line_id, sort inside each group, number the rows 1, 2, 3...
w = Window.partitionBy("line_id").orderBy("order_id", "product_id")

numbered = clean_lines.withColumn("rn", F.row_number().over(w))

# One duplicated line_id: two rows, numbered 1 and 2
numbered.filter(F.col("line_id") == "L0001_1").show()

In [ ]:
kept    = numbered.filter(F.col("rn") == 1).drop("rn")   # one row per line_id
removed = numbered.filter(F.col("rn") > 1)               # the extra copies, kept for audit

print("kept   :", kept.count())
print("removed:", removed.count())

### Task 5 · Corrections: keep the latest version

In real data, a "duplicate" is often a **correction**: the same customer arrives again with a new email. Now it matters which row wins.

**Problem:** A small test table:
- `C1` has an old and a new email.
- `C2` has two versions with the **same** `updated_at`.

Keep the **latest** version of each customer.

**Expected:**
- `dropDuplicates(["customer_id"])` keeps **any** row. It has no idea what "latest" means, and the result can change between runs.
- Sorting by `updated_at` only: for **C2** the two rows are tied, so the winner isn't guaranteed.
- Sorting by `updated_at` desc, then `batch_id` desc (the **tie-break**: the later load wins): **C1 → `new@example.test`** and **C2 → `b@example.test`**, every time.

In [ ]:
updates = spark.createDataFrame([
    ("C1", "old@example.test", "2026-06-01 10:00:00", 1),
    ("C1", "new@example.test", "2026-06-05 10:00:00", 2),   # newer -> should win
    ("C2", "a@example.test",   "2026-06-03 09:00:00", 1),
    ("C2", "b@example.test",   "2026-06-03 09:00:00", 2),   # same time as above -> needs a tie-break
], "customer_id STRING, email STRING, updated_at STRING, batch_id INT")

updates = updates.withColumn("updated_at", F.to_timestamp("updated_at"))
updates.show(truncate=False)

In [ ]:
# dropDuplicates keeps ANY one row per customer: there is no "latest" rule
updates.dropDuplicates(["customer_id"]).show(truncate=False)

In [ ]:
# Sort by updated_at only: C2's two rows are tied, so either one may get rn = 1
w_time_only = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())

updates.withColumn("rn", F.row_number().over(w_time_only)).show(truncate=False)

In [ ]:
# Deterministic rule: newest updated_at first; if tied, the higher batch_id first
w_latest = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc(), F.col("batch_id").desc())

latest = (updates
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .drop("rn"))

latest.show(truncate=False)

### Task 6 · Publish the clean lines and check everything adds up

**Problem:**
1. Add the Day 03 money columns (`line_gross`, `line_net`) and save the unique lines as **`clean_lines`** for the coming days. Save the removed copies too.
2. Check that `line_id` is now unique.
3. Check that every raw line is accounted for: **raw = quarantine (Day 04) + removed duplicates + clean unique lines**.
4. Run the dedup a second time. Does anything change?

**Expected:**
1. Two Parquet folders in `scratch/day05`.
2. **767** rows and **767** distinct `line_id`s.
3. **785 = 10 + 8 + 767** → `True`. The 767 also matches the manifest's `clean_unique_lines`.
4. Still **767**. Running it again changes nothing, which is called **idempotent**.

In [ ]:
# Add the Day 03 money columns
clean_unique = (kept
    .withColumn("line_gross", F.col("quantity") * F.col("unit_price"))
    .withColumn("line_net", F.round(F.col("line_gross") * (1 - F.col("discount_pct")), 2).cast("decimal(12,2)")))

clean_unique.write.mode("overwrite").parquet(f"{OUT}/clean_lines")        # used from Day 06 onwards
removed.write.mode("overwrite").parquet(f"{OUT}/removed_duplicates")      # audit trail

In [ ]:
published = spark.read.parquet(f"{OUT}/clean_lines")

# line_id is now unique: the row count equals the number of distinct line_ids
print("rows             :", published.count())
print("distinct line_ids:", published.select("line_id").distinct().count())

In [ ]:
# Every raw line ends up in exactly one place
raw_count        = spark.read.option("header", True).csv(f"{RAW}/order_items.csv").count()
quarantine_count = spark.read.parquet(f"{SCRATCH}/day04/quarantine_lines").count()
removed_count    = spark.read.parquet(f"{OUT}/removed_duplicates").count()
clean_count      = published.count()

print("raw        :", raw_count)
print("quarantine :", quarantine_count)
print("removed    :", removed_count)
print("clean      :", clean_count, "| manifest clean_unique_lines:", manifest["clean_unique_lines"])
print("raw == quarantine + removed + clean ?", raw_count == quarantine_count + removed_count + clean_count)

In [ ]:
# Idempotent: deduplicating the already-clean data changes nothing
again = published.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1)
print("after a second dedup:", again.count())

### Key takeaways
- **Grain first.** Know what one row means. In `order_items` it's one **line**, so `line_id` is the key, and a repeated `order_id` is normal.
- **Exact duplicate vs key duplicate:**
  - An exact duplicate is identical in every column (`distinct()`).
  - A key duplicate shares the key but may differ in other columns (`dropDuplicates([key])`).
- **The wrong key loses data:** `dropDuplicates(["order_id"])` dropped 467 real lines.
- **Window + `row_number`** is the standard way to keep one row per key: `partitionBy` the key, `orderBy` your rule, and keep `rn = 1`.
- **Deterministic = the ordering never ties.** A timestamp alone can tie, so add a tie-break column (batch, load time, or a sequence number).
- **`dropDuplicates` picks any row.** Never rely on it to keep the latest version.
- **Keep the removed rows** for audit, and reconcile: raw = quarantine + removed + clean.
- **Clean before you dedup.** `" L1"` and `"L1"` look different until trimmed. Reject missing keys first, or all the null keys would collapse into one row.

*Skipped on purpose (they don't add a new concept): the repartition-stability test, the configurable dedup function, and the policy for data with no ordering column. Streaming deduplication comes on Day 24.*